# 05 — LightGBM Baseline on Simple Image Statistics

This notebook reproduces the **LightGBM Robust Version** used as the classical
machine-learning baseline in:

> **A Dual Representation Framework for Malicious QR Code Detection Using Fused Feature Learning and Deep Visual Modeling**

The implementation is derived from the original experiment section titled:

> **TRAIN LIGHTGBM ON SIMPLE IMAGE STATISTICS (ROBUST VERSION)**

It does **not** use the earlier ResNet-18 deep-feature LightGBM experiment.

## Pipeline

1. Load the controlled train, validation, and test metadata.
2. Extract simple image statistics from each QR-code image.
3. Train LightGBM using the original robust configuration.
4. Evaluate the untouched test set.
5. Save metrics, predictions, confusion matrix, training history, model, and feature importance.

## 1. Imports and Reproducibility

Install LightGBM when needed:

```bash
pip install lightgbm
```

In [ ]:
import json
import os
import random
from pathlib import Path
from time import perf_counter

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from tqdm.auto import tqdm

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("LightGBM version:", lgb.__version__)
print("Random seed     :", SEED)

## 2. Repository Paths

In [ ]:
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebooks" else CURRENT_DIR

PROCESSED_DATA_DIR = PROJECT_ROOT / "Data" / "processed"

MODEL_DIR = PROJECT_ROOT / "Models" / "lightgbm"
RESULT_DIR = PROJECT_ROOT / "Results" / "lightgbm"
FIGURES_DIR = RESULT_DIR / "figures"
TABLES_DIR = RESULT_DIR / "tables"
METRICS_DIR = RESULT_DIR / "metrics"
FEATURE_DIR = PROCESSED_DATA_DIR / "lightgbm_image_stats"

for directory in [
    MODEL_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    METRICS_DIR,
    FEATURE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = PROCESSED_DATA_DIR / "train.csv"
VAL_CSV = PROCESSED_DATA_DIR / "val.csv"
TEST_CSV = PROCESSED_DATA_DIR / "test.csv"

print("Project root:", PROJECT_ROOT)
print("Results     :", RESULT_DIR)

## 3. Load Controlled Dataset Partitions

Notebook 01 stores repository-relative paths in `image_path`. This notebook normalizes
that field to the original LightGBM interface, `image`, without changing the split membership.

In [ ]:
for path in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path}. Run Notebook 01 first."
        )

def load_split(path: Path, split_name: str) -> pd.DataFrame:
    dataframe = pd.read_csv(path).copy()

    if "image_path" in dataframe.columns and "image" not in dataframe.columns:
        dataframe = dataframe.rename(columns={"image_path": "image"})

    required = {"image", "label"}
    missing = required - set(dataframe.columns)
    if missing:
        raise ValueError(
            f"{split_name} split is missing columns: {sorted(missing)}"
        )

    dataframe["image"] = dataframe["image"].astype(str).str.strip()
    dataframe["label"] = dataframe["label"].astype(str).str.strip().str.lower()

    invalid_labels = ~dataframe["label"].isin(["benign", "malicious"])
    if invalid_labels.any():
        raise ValueError(
            f"{split_name} split contains invalid labels: "
            f"{dataframe.loc[invalid_labels, 'label'].unique().tolist()}"
        )

    return dataframe.reset_index(drop=True)

train_df = load_split(TRAIN_CSV, "train")
val_df = load_split(VAL_CSV, "validation")
test_df = load_split(TEST_CSV, "test")

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

## 4. Extract Simple Image-Statistics Features

The original robust LightGBM baseline uses eleven descriptors:

- width;
- height;
- file size;
- mean intensity;
- intensity standard deviation;
- minimum intensity;
- maximum intensity;
- black-pixel ratio;
- white-pixel ratio;
- Shannon entropy;
- edge density derived from simple horizontal and vertical gradients.

In [ ]:
def compute_entropy(arr_uint8: np.ndarray) -> float:
    histogram = np.bincount(
        arr_uint8.flatten(),
        minlength=256,
    ).astype(np.float64)

    probabilities = histogram / histogram.sum()
    probabilities = probabilities[probabilities > 0]

    return float(
        -np.sum(probabilities * np.log2(probabilities))
    )

def extract_image_stats(relative_path: str) -> dict:
    absolute_path = PROJECT_ROOT / relative_path

    try:
        with Image.open(absolute_path) as image:
            grayscale = image.convert("L")
            array = np.asarray(grayscale)

        height, width = array.shape
        file_size = os.path.getsize(absolute_path)

        gradient_x = np.abs(
            np.diff(array.astype(np.float32), axis=1)
        )
        gradient_y = np.abs(
            np.diff(array.astype(np.float32), axis=0)
        )

        edge_pixels_x = (
            (gradient_x > 20).mean()
            if gradient_x.size > 0
            else 0.0
        )
        edge_pixels_y = (
            (gradient_y > 20).mean()
            if gradient_y.size > 0
            else 0.0
        )

        return {
            "image": relative_path,
            "width": width,
            "height": height,
            "file_size": file_size,
            "mean_intensity": float(array.mean()),
            "std_intensity": float(array.std()),
            "min_intensity": int(array.min()),
            "max_intensity": int(array.max()),
            "black_ratio": float((array < 128).mean()),
            "white_ratio": float((array >= 128).mean()),
            "entropy": compute_entropy(array.astype(np.uint8)),
            "edge_density": float(
                (edge_pixels_x + edge_pixels_y) / 2.0
            ),
            "ok": 1,
        }

    except Exception:
        return {
            "image": relative_path,
            "width": np.nan,
            "height": np.nan,
            "file_size": np.nan,
            "mean_intensity": np.nan,
            "std_intensity": np.nan,
            "min_intensity": np.nan,
            "max_intensity": np.nan,
            "black_ratio": np.nan,
            "white_ratio": np.nan,
            "entropy": np.nan,
            "edge_density": np.nan,
            "ok": 0,
        }

def build_stats_dataframe(
    split_df: pd.DataFrame,
    split_name: str,
) -> pd.DataFrame:
    rows = [
        extract_image_stats(image_path)
        for image_path in tqdm(
            split_df["image"],
            desc=f"Extracting {split_name} statistics",
        )
    ]

    feature_df = pd.DataFrame(rows)
    return split_df.merge(feature_df, on="image", how="left")

train_stats_df = build_stats_dataframe(train_df, "train")
val_stats_df = build_stats_dataframe(val_df, "validation")
test_stats_df = build_stats_dataframe(test_df, "test")

print("Train readable:", int(train_stats_df["ok"].sum()), "/", len(train_stats_df))
print("Val readable  :", int(val_stats_df["ok"].sum()), "/", len(val_stats_df))
print("Test readable :", int(test_stats_df["ok"].sum()), "/", len(test_stats_df))

## 5. Save Extracted Baseline Features

In [ ]:
TRAIN_FEATURE_CSV = FEATURE_DIR / "train_image_stats.csv"
VAL_FEATURE_CSV = FEATURE_DIR / "val_image_stats.csv"
TEST_FEATURE_CSV = FEATURE_DIR / "test_image_stats.csv"

train_stats_df.to_csv(TRAIN_FEATURE_CSV, index=False)
val_stats_df.to_csv(VAL_FEATURE_CSV, index=False)
test_stats_df.to_csv(TEST_FEATURE_CSV, index=False)

print("Saved:")
print(" -", TRAIN_FEATURE_CSV)
print(" -", VAL_FEATURE_CSV)
print(" -", TEST_FEATURE_CSV)

## 6. Source-of-Truth Hyperparameters

These are the exact settings found in the robust LightGBM experiment used for the paper.

| Hyperparameter | Value |
|---|---:|
| Objective | Binary |
| Number of estimators | 500 |
| Learning rate | 0.05 |
| Number of leaves | 31 |
| Maximum depth | −1 |
| Subsample | 0.8 |
| Feature fraction (`colsample_bytree`) | 0.8 |
| Random seed | 42 |
| CPU jobs | −1 |
| Evaluation metric | Binary log loss |
| Missing-value handling | Filled with 0 |

In [ ]:
LIGHTGBM_HYPERPARAMETERS = {
    "objective": "binary",
    "n_estimators": 500,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": SEED,
    "n_jobs": -1,
    "evaluation_metric": "binary_logloss",
    "missing_value_fill": 0,
}

with (MODEL_DIR / "lightgbm_hyperparameters.json").open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(LIGHTGBM_HYPERPARAMETERS, file, indent=2)

print(LIGHTGBM_HYPERPARAMETERS)

## 7. Prepare the Robust Feature Matrices

The original implementation includes a defensive column resolver because earlier
metadata joins could produce names such as `width_x` and `width_y`.
That behavior is preserved here.

In [ ]:
for dataframe in [train_stats_df, val_stats_df, test_stats_df]:
    if "ok" in dataframe.columns:
        dataframe.drop(
            dataframe[dataframe["ok"] != 1].index,
            inplace=True,
        )

label_map = {"benign": 0, "malicious": 1}

for dataframe in [train_stats_df, val_stats_df, test_stats_df]:
    dataframe["label_id"] = dataframe["label"].map(label_map)

    if dataframe["label_id"].isna().any():
        raise ValueError("Label encoding produced missing values.")

    dataframe["label_id"] = dataframe["label_id"].astype(int)

def pick_existing_column(
    dataframe: pd.DataFrame,
    candidates: list[str],
):
    for candidate in candidates:
        if candidate in dataframe.columns:
            return candidate
    return None

resolved_features = {
    "width": pick_existing_column(
        train_stats_df,
        ["width", "width_y", "width_x"],
    ),
    "height": pick_existing_column(
        train_stats_df,
        ["height", "height_y", "height_x"],
    ),
    "file_size": pick_existing_column(
        train_stats_df,
        ["file_size"],
    ),
    "mean_intensity": pick_existing_column(
        train_stats_df,
        ["mean_intensity"],
    ),
    "std_intensity": pick_existing_column(
        train_stats_df,
        ["std_intensity"],
    ),
    "min_intensity": pick_existing_column(
        train_stats_df,
        ["min_intensity"],
    ),
    "max_intensity": pick_existing_column(
        train_stats_df,
        ["max_intensity"],
    ),
    "black_ratio": pick_existing_column(
        train_stats_df,
        ["black_ratio"],
    ),
    "white_ratio": pick_existing_column(
        train_stats_df,
        ["white_ratio"],
    ),
    "entropy": pick_existing_column(
        train_stats_df,
        ["entropy"],
    ),
    "edge_density": pick_existing_column(
        train_stats_df,
        ["edge_density"],
    ),
}

missing_features = [
    name
    for name, source_column in resolved_features.items()
    if source_column is None
]

if missing_features:
    raise ValueError(
        f"Missing required features: {missing_features}"
    )

feature_cols = list(resolved_features.keys())

def build_feature_dataframe(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    output = pd.DataFrame(index=dataframe.index)

    for final_name, source_name in resolved_features.items():
        output[final_name] = dataframe[source_name]

    return output

X_train = build_feature_dataframe(train_stats_df).fillna(0)
X_val = build_feature_dataframe(val_stats_df).fillna(0)
X_test = build_feature_dataframe(test_stats_df).fillna(0)

y_train = train_stats_df["label_id"]
y_val = val_stats_df["label_id"]
y_test = test_stats_df["label_id"]

print("Resolved features:", feature_cols)
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

## 8. Train the Robust LightGBM Baseline

In [ ]:
evaluation_history = {}

lightgbm_model = lgb.LGBMClassifier(
    objective="binary",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    n_jobs=-1,
)

lightgbm_model.fit(
    X_train,
    y_train,
    eval_set=[
        (X_train, y_train),
        (X_val, y_val),
    ],
    eval_names=["train", "val"],
    eval_metric="binary_logloss",
    callbacks=[
        lgb.record_evaluation(evaluation_history)
    ],
)

MODEL_PATH = MODEL_DIR / "lightgbm_image_statistics.joblib"
joblib.dump(lightgbm_model, MODEL_PATH)

print("Training completed.")
print("Saved model:", MODEL_PATH)

## 9. Test Evaluation and Inference Time

In [ ]:
inference_start = perf_counter()
y_proba = lightgbm_model.predict_proba(X_test)[:, 1]
inference_seconds = perf_counter() - inference_start

y_pred = lightgbm_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(
    y_test,
    y_pred,
    zero_division=0,
)
recall = recall_score(
    y_test,
    y_pred,
    zero_division=0,
)
f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0,
)
roc_auc = roc_auc_score(y_test, y_proba)
matrix = confusion_matrix(y_test, y_pred)

metrics_df = pd.DataFrame(
    [
        {
            "model": "LightGBM_ImageStats_Robust",
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "roc_auc": roc_auc,
            "inference_seconds": inference_seconds,
            "milliseconds_per_sample": (
                inference_seconds / len(X_test) * 1000
            ),
        }
    ]
)

METRICS_PATH = METRICS_DIR / "lightgbm_test_metrics.csv"
metrics_df.to_csv(METRICS_PATH, index=False)

predictions_df = test_stats_df[
    ["image", "label", "label_id"]
].copy()
predictions_df["predicted_label"] = y_pred
predictions_df["prob_malicious"] = y_proba
predictions_df.to_csv(
    TABLES_DIR / "lightgbm_test_predictions.csv",
    index=False,
)

confusion_df = pd.DataFrame(
    matrix,
    index=["true_benign", "true_malicious"],
    columns=["pred_benign", "pred_malicious"],
)
confusion_df.to_csv(
    TABLES_DIR / "lightgbm_confusion_matrix.csv"
)

report = classification_report(
    y_test,
    y_pred,
    target_names=["benign", "malicious"],
    zero_division=0,
    digits=4,
)

with (METRICS_DIR / "lightgbm_classification_report.txt").open(
    "w",
    encoding="utf-8",
) as file:
    file.write(report)

print(metrics_df)
print()
print(report)

## 10. Save Training History

In [ ]:
history_rows = []

train_logloss = evaluation_history["train"]["binary_logloss"]
val_logloss = evaluation_history["val"]["binary_logloss"]

for iteration, (
    train_value,
    val_value,
) in enumerate(
    zip(train_logloss, val_logloss),
    start=1,
):
    history_rows.append(
        {
            "iteration": iteration,
            "train_logloss": train_value,
            "val_logloss": val_value,
        }
    )

history_df = pd.DataFrame(history_rows)
history_df.to_csv(
    TABLES_DIR / "lightgbm_training_history.csv",
    index=False,
)

display(history_df.head())

## 11. Training and Validation Log-Loss Curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    history_df["iteration"],
    history_df["train_logloss"],
    label="Train Log Loss",
)
plt.plot(
    history_df["iteration"],
    history_df["val_logloss"],
    label="Validation Log Loss",
)
plt.xlabel("Iteration")
plt.ylabel("Binary Log Loss")
plt.title(
    "LightGBM Image-Statistics Training and Validation Log Loss"
)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "lightgbm_logloss_curves.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 12. Test Confusion Matrix

In [ ]:
plt.figure(figsize=(6, 5))

display_object = ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=["Benign", "Malicious"],
)
display_object.plot(
    values_format="d",
    colorbar=False,
)

plt.title("LightGBM Image-Statistics Confusion Matrix")
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "lightgbm_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 13. Feature Importance

In [ ]:
importance_df = pd.DataFrame(
    {
        "feature": feature_cols,
        "importance": lightgbm_model.feature_importances_,
    }
).sort_values(
    "importance",
    ascending=False,
)

importance_df.to_csv(
    TABLES_DIR / "lightgbm_feature_importance.csv",
    index=False,
)

plt.figure(figsize=(8, 6))
plt.barh(
    importance_df["feature"][::-1],
    importance_df["importance"][::-1],
)
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title(
    "LightGBM Image-Statistics Feature Importance"
)
plt.grid(True)
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "lightgbm_feature_importance.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

display(importance_df)

## 14. Final Summary

In [ ]:
assert len(feature_cols) == 11
assert not X_train.isna().any().any()
assert not X_val.isna().any().any()
assert not X_test.isna().any().any()

print("=" * 68)
print("LIGHTGBM ROBUST BASELINE COMPLETED")
print("=" * 68)
display(metrics_df)
print("Features:", feature_cols)
print("Model   :", MODEL_PATH)
print("Results :", RESULT_DIR)
print("=" * 68)